# Domain shift / degradation trend analysis

Questa analisi non usa il modello ST-GNN e non richiede training. Misura se la produzione reale degli impianti cala nel tempo rispetto a una baseline meteo/PVGIS.

- `pr_pvgis = ENERGIA reale / (kWp * PVGIS_POA)`: performance normalizzata rispetto a quanto ci si aspetta dato l'irraggiamento.
- `pr_month_z`: versione destagionalizzata mese per mese, cioe' confronto del PR giornaliero con media e deviazione standard del relativo mese.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
OUT = ROOT / 'outputs' / 'domain_shift_trend'
OUT

## Run dello script

Esegui questa cella per rigenerare CSV, summary e grafici. Se hai gia' generato gli output, puoi saltarla e caricare direttamente le celle successive.

In [ ]:
cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'analyze_domain_shift_trend.py'),
    '--out-dir', str(OUT),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

## Summary numerico

In [ ]:
with open(OUT / 'summary.json', encoding='utf-8') as f:
    summary = json.load(f)

summary

In [ ]:
fleet_trends = pd.read_csv(OUT / 'fleet_trend_summary.csv')
plant_trends = pd.read_csv(OUT / 'plant_trend_summary.csv')
fleet_monthly = pd.read_csv(OUT / 'fleet_monthly_performance.csv', parse_dates=['date'])
plant_monthly = pd.read_csv(OUT / 'plant_monthly_performance.csv', parse_dates=['date'])

fleet_trends

## Grafici generati dallo script

In [ ]:
for name in ['fleet_pvgis_pr_trend.png', 'top_decreasing_plants_pvgis_pr.png']:
    path = OUT / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Missing: {path}')

## Impianti con trend decrescente piu' forte

In [ ]:
cols = [
    'metric', 'plant', 'plant_id', 'n_points', 'mean_value',
    'slope_per_year', 'relative_change_pct_per_year', 'p_value',
    'kendall_tau', 'decreasing'
]

pr_trends = plant_trends[plant_trends['metric'] == 'pr_pvgis_monthly'].copy()
pr_trends.sort_values('slope_per_year')[cols].head(20)

## Andamento fleet mensile

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fleet_monthly['date'], fleet_monthly['weighted_pr_pvgis'], 'o-', label='weighted PR PVGIS')
ax.plot(fleet_monthly['date'], fleet_monthly['median_pr_pvgis'], 's--', label='median plant PR')
ax.axhline(fleet_monthly['weighted_pr_pvgis'].mean(), color='0.3', linewidth=1, alpha=0.6)
ax.set_xlabel('Mese')
ax.set_ylabel('actual / (kWp * PVGIS POA)')
ax.set_title('Performance normalizzata PVGIS - flotta')
ax.legend()
plt.tight_layout()

## Lettura per la tesi

Un trend negativo di `pr_pvgis` indica che, a parita' di riferimento meteo/PVGIS e capacita' stimata o reale, la produzione cala nel tempo. Questo e' il segnale piu' vicino a soiling, degrado sistemico o domain shift temporale della flotta.

`pr_month_z` serve invece a ridurre la stagionalita': ogni giorno viene confrontato con la distribuzione del suo mese. Con un solo anno il risultato va letto come diagnostica esplorativa; con dati multi-anno diventa molto piu' solido.